# InfiniteTalk｜画像＋音声 → 40秒アバター動画（Colab修正版 v5・FlashAttention代替＋Ninja修復＋ログ診断）

Colabの **Python 3.13 と InfiniteTalkの旧依存パッケージの不一致** を回避するため、ノートブックのカーネルとは別に **Python 3.10専用環境** を自動構築します。

- **用途**：画像1枚＋音声1ファイルから約40秒のアバター動画を生成する
- **GPU**：G4推奨。A100 80GBも候補。A100 40GBは省VRAMモード（遅くなる）
- **方式**：InfiniteTalk公式スクリプト + 量子化済みINT8モデル（高速化LoRA入り）+ FP8テキストエンコーダー
- **使い方**：①→②→②-B→③→④→⑤→⑥→⑦ の順に実行。**2回目以降は⑤〜⑦のみ**

### 前版からの修正

1. `numpy==1.26.4` が動作しない Python 3.13環境へ直接インストールしない
2. 公式コードに残る `inspect.ArgSpec` の互換性問題を修正
3. Wav2Vec2の注意点（eager attention）を修正
4. インストールのログを表示し、失敗時は**どのパッケージが原因か分かる**ように変更

⚠ **注意：** ノートブックのPython構文とセルの整合性は検証しましたが、Colabの実GPUでのモデル取得・40秒生成は未実行です。最初は `DURATION_SEC=5` で動作確認を推奨します。GPUとストレージを多く使用します。

公式：https://github.com/MeiGen-AI/InfiniteTalk


### v3での改善
生成が終了コード1で失敗した場合に**元エラーログの重要部分を自動表示**します。エラーの原因が判明するまで、CUDAメモリ不足と決めつけません。GPU・ディスク・ファイルサイズの事前診断も追加しました。

- **v4修正**：FP8 T5で必要な Ninja を専用Python 3.10環境に導入し、生成サブプロセスでも確実に実行できるよう PATH に追加。CUDA拡張のコンパイル負荷も制限。


### v5での修正
- G4で `AssertionError: FLASH_ATTN_2_AVAILABLE` が発生する問題に対し、PyTorch標準SDPAを代替として利用（FlashAttention 2のインストール不要）。
- CLIPだけでなく、Wan本体のFlashAttention呼び出しも同一の互換関数で補います。
- FP16/BF16の入出力型、Attentionマスクと長さの扱いを維持し、GPU上の簡易実行テストをします。
- FlashAttentionとは計算実装が異なるため、速度や画質は完全一致を保証しません。


## ① GPU・空き容量を確認

In [ ]:
import subprocess, shutil, re, os, sys

def cmd_out(cmd):
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()

try:
    gpus = cmd_out(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'])
except Exception as e:
    raise RuntimeError('GPUが有効ではありません。Colabの「ランタイムのタイプを変更」でG4またはA100を選んでください。') from e
print('GPU:', gpus)
mem_mib = int(gpus.splitlines()[0].rsplit(',', 1)[-1].strip())
mem_gib = mem_mib / 1024
free_gib = shutil.disk_usage('/content').free / (1024**3)
print(f'VRAM: {mem_gib:.1f} GiB / ディスク空き: {free_gib:.1f} GiB')
print('Python:', sys.version.split()[0])
if mem_gib < 38:
    raise RuntimeError('この14Bモデルの40秒生成にはVRAMが不足しやすいです。G4またはA100を選んでください。')
if free_gib < 42:
    raise RuntimeError('モデル取得に必要なディスク空きが足りません。45〜55GiB程度の空きを確保してください。')
if mem_gib < 55:
    print('⚠ 40GiB前後のGPU：モデルをCPUへ退避する低VRAMモードを自動使用します。速度が低下します。')
print('✅ 事前チェック完了')

## ② Colab環境と分離したPython 3.10を構築（初回のみ）

**ColabのPython 3.13は変更しません。** `uv`を使ってPython 3.10仮想環境を作り、InfiniteTalk専用に必要なライブラリをインストールします。

エラー時は下部のインストールログに実際の原因が表示されます。**古いノートブックで途中失敗していても、そのままこのセルを実行できます。** これまでのコード取得・モデルダウンロードが残っていれば再利用します。


In [ ]:
import os, sys, subprocess, pathlib, shutil, textwrap
ROOT = pathlib.Path('/content/InfiniteTalk')
VENV = pathlib.Path('/content/infinitetalk_py310')
VENV_PY = VENV / 'bin' / 'python'

# ログを省略しない。パッケージ名・依存エラーが見えるようにする。
def run(command, *, cwd=None):
    command = list(map(str, command))
    print('▶', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

# 初期化: 以前のColab Python 3.13向けpip失敗は無視して独立環境を使う。
os.environ['PIP_NO_CACHE_DIR'] = '1'
if not (ROOT / 'generate_infinitetalk.py').is_file():
    run(['git','clone','--depth','1','https://github.com/MeiGen-AI/InfiniteTalk.git',ROOT])
else:
    print('✅ 既存のInfiniteTalkリポジトリを再利用')

# uv はColabカーネルの Python 3.13 にインストールしても、モデル環境を汚さない。
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'uv'])
UV = shutil.which('uv')
assert UV, 'uvのインストールを確認できません。'

if not VENV_PY.exists():
    run([UV, 'python', 'install', '3.10'])
    run([UV, 'venv', '--python', '3.10', str(VENV)])
print('専用Python:', subprocess.check_output([str(VENV_PY), '-V'], text=True).strip())

# PyTorch/xFormersのABI/CUDA互換性を揃える（G4 Blackwell向けCUDA 12.8）。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '--index-url', 'https://download.pytorch.org/whl/cu128',
     'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1'])

# ColabではGradio UIやDashscope APIを使わないため取り除き、依存関係の将来変更を回避。
# xfuserは単一GPUでも一部の公式コードからimportされるので2025年互換版を維持。
REQ = ROOT / 'requirements.colab.txt'
REQ.write_text('\n'.join(line for line in (ROOT / 'requirements.txt').read_text().splitlines()
                       if not line.strip().startswith(('gradio', 'dashscope', 'xfuser'))) + '\n')
# torchの版を固定し、インストール時のGPUパッケージ入れ替えを防ぐ。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '-r', str(REQ), 'xfuser==0.4.4',
     'torch==2.7.1', 'xformers==0.0.31.post1',
     'numpy==1.26.4', 'opencv-python==4.10.0.84',
     'transformers==4.49.0', 'diffusers==0.33.1',
     'huggingface_hub==0.34.4', 'librosa==0.10.2.post1',
     'misaki[en]', 'soundfile', 'hf-xet', 'ninja'])

# FP8 T5をGPUに転送する際、optimum-quantoがC++/CUDA拡張をJITビルドする。
# uvが作った仮想環境にはpipが入っていない場合があるため、必ずuv pipでインストールする。
# venv/binを PATH の先頭に追加し、torch.utils.cpp_extension がninjaを見つけられるようにする。
os.environ['PATH'] = str(VENV / 'bin') + os.pathsep + os.environ.get('PATH', '')
run(['ninja', '--version'])
print('✅ Ninja を発見。CUDA拡張のビルドに利用できます。')
# nvccがない場合は、次のGPU生成セルでエラーになる可能性がある。ここで状況を可視化。
if shutil.which('nvcc'):
    result = subprocess.run(['nvcc', '--version'], text=True, capture_output=True)
    print('CUDA compiler:', (result.stdout.strip().splitlines() or ['unknown'])[-1])
else:
    print('⚠️ nvccが PATH に見つかりません。次にCUDA toolkit関連のエラーが出る可能性があります。')

if shutil.which('ffmpeg') is None:
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','ffmpeg'])

# Pythonのバージョン変更に対応する公式ソースの最小限の互換性パッチ。
# パッチは繰り返し実行しても変化しない（冪等）。
mt = ROOT / 'wan' / 'multitalk.py'
s = mt.read_text()
if 'from inspect import ArgSpec' in s:
    s = s.replace('from inspect import ArgSpec', 'from inspect import FullArgSpec as ArgSpec')
    mt.write_text(s)
    print('✅ inspect.ArgSpec互換パッチを適用')
wv = ROOT / 'src' / 'audio_analysis' / 'wav2vec2.py'
s = wv.read_text()
needle = '        self.config.output_attentions = True'
if needle in s and '_attn_implementation = "eager"' not in s:
    s = s.replace(needle, '        self.config._attn_implementation = "eager"\n' + needle)
    wv.write_text(s)
    print('✅ Wav2Vec2 eager attention互換パッチを適用')

# エラーを早めに検出。OSカーネルではなく専用Pythonで実行。
run([str(VENV_PY), '-c',
     'import torch, torchvision, xformers, transformers, diffusers, numpy; '
     'import wan.multitalk; '
     'print("環境OK: torch", torch.__version__, "cuda", torch.version.cuda, "numpy", numpy.__version__)'],
    cwd=ROOT)
print('✅ 環境構築完了。Colabランタイムの再起動は不要です。')


## ②-B FlashAttentionを必要としないGPU互換パッチ（初回のみ）

**②の直後に実行してください。** 既存のv4でモデルを取得済みの場合、下のパッチセルだけ今のColabに追加して実行し、⑤→⑥→⑦を再実行すればOKです。

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
attn_file = ROOT / 'wan' / 'modules' / 'attention.py'
assert attn_file.is_file(), '②でInfiniteTalkをセットアップしてください。'
assert VENV_PY.is_file(), '②でPython 3.10環境を構築してください。'

s = attn_file.read_text(encoding='utf-8')
marker = '# COLAB_SDPA_FALLBACK_V5'
needle = '    half_dtypes = (torch.float16, torch.bfloat16)'

fallback = '''    # COLAB_SDPA_FALLBACK_V5
    # FlashAttention 2 is unavailable in some Colab runtimes (including G4).
    # Use PyTorch SDPA while preserving the [B, seq, heads, dim] API and dtype.
    if not FLASH_ATTN_2_AVAILABLE and (version == 2 or not FLASH_ATTN_3_AVAILABLE):
        b, lq, n, d = q.shape
        lk = k.shape[1]
        out_dtype = q.dtype
        work_dtype = v.dtype if v.dtype in (torch.float16, torch.bfloat16) else dtype
        qq = q.transpose(1, 2).to(dtype=work_dtype)
        kk = k.transpose(1, 2).to(dtype=work_dtype)
        vv = v.transpose(1, 2).to(dtype=work_dtype)
        if q_scale is not None:
            qq = qq * q_scale

        mask = None
        if k_lens is not None:
            kl = torch.as_tensor(k_lens, device=q.device).reshape(b)
            valid_keys = torch.arange(lk, device=q.device)[None, None, None, :] < kl[:, None, None, None]
            mask = valid_keys
        if window_size is not None and tuple(window_size) != (-1, -1):
            left, right = window_size
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            local = torch.ones((1, 1, lq, lk), device=q.device, dtype=torch.bool)
            if left >= 0:
                local = local & (ki >= qi - left)
            if right >= 0:
                local = local & (ki <= qi + right)
            mask = local if mask is None else mask & local
        # Build an explicit causal mask when necessary to combine with key masking.
        if causal:
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            # FlashAttention varlen causal alignment is bottom-right for Lq != Lk.
            causal_mask = ki <= qi + (lk - lq)
            mask = causal_mask if mask is None else mask & causal_mask

        result = torch.nn.functional.scaled_dot_product_attention(
            qq, kk, vv, attn_mask=mask, dropout_p=dropout_p,
            is_causal=False, scale=softmax_scale)
        if q_lens is not None:
            ql = torch.as_tensor(q_lens, device=q.device).reshape(b)
            valid_q = torch.arange(lq, device=q.device)[None, None, :, None] < ql[:, None, None, None]
            result = result.masked_fill(~valid_q, 0)
        return result.transpose(1, 2).contiguous().to(out_dtype)

'''

if marker in s:
    print('✅ SDPAパッチは既に適用済み（再実行しても問題ありません）')
else:
    assert needle in s, 'attention.pyのコード構造が変わっています。パッチ適用を中止しました。'
    s = s.replace(needle, fallback + needle, 1)
    # Syntax validation before modifying checked-out source
    compile(s, str(attn_file), 'exec')
    attn_file.with_suffix('.py.before_colab_sdpa_v5').write_text(attn_file.read_text(encoding='utf-8'), encoding='utf-8')
    attn_file.write_text(s, encoding='utf-8')
    print('✅ FlashAttention不在時のPyTorch SDPAフォールバックを追加')

# New subprocess: no runtime restart. Two tests including key padding and causal behavior.
probe = r'''
import torch
from wan.modules.attention import flash_attention
assert torch.cuda.is_available(), 'GPUが有効ではありません'
dev = 'cuda'
for lq, lk, causal in [(8, 8, False), (5, 9, True)]:
    q = torch.randn(2, lq, 4, 32, device=dev, dtype=torch.float16)
    k = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    v = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    k_lens = torch.tensor([lk, lk - 1], device=dev, dtype=torch.int32)
    out = flash_attention(q, k, v, k_lens=k_lens, causal=causal, version=2)
    assert out.shape == q.shape and out.dtype == q.dtype
    assert torch.isfinite(out).all()
print('✅ Attention動作テスト成功 / GPU:', torch.cuda.get_device_name(0))
'''
subprocess.run([str(VENV_PY), '-c', probe], cwd=str(ROOT), check=True)
print('✅ このあと⑤で画像と音声を用意し、⑥→⑦を実行してください。')


## ③ 量子化モデルをダウンロード（初回のみ）

ダウンロードには **②で構築した専用Python** を使います。以前のダウンロードが残っていれば再利用します。必要なベースモデル付属ファイル・量子化DiT・量子化T5・Wav2Vec2のみを取得します（数十GB規模）。

途中切断はこのセルを再実行して再開。Hugging Face上のモデルが削除・非公開化されている場合はエラーとして停止します。


In [ ]:
from pathlib import Path
import os, subprocess, shutil, textwrap
ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '②の環境構築から実行してください。'
W = ROOT / 'weights'
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
QUANT_NAME = 'infinitetalk_single_int8_lora'
QUANT = IT / 'quant_models' / (QUANT_NAME + '.safetensors')

free = shutil.disk_usage('/content').free / 1024**3
if not QUANT.exists() and free < 32:
    raise RuntimeError(f'ディスク空き {free:.1f} GiB。容量不足の可能性。')

# HFライブラリも仮想環境側。ノートブックカーネルへインポートしない。
download_script = r'''
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download
W = Path('/content/InfiniteTalk/weights')
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
print('1/3: Wan2.1補助モデル', flush=True)
snapshot_download('Wan-AI/Wan2.1-I2V-14B-480P',local_dir=str(BASE),
  allow_patterns=['config.json','Wan2.1_VAE.pth',
   'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
   'google/**','xlm-roberta-large/**'],max_workers=3)
print('2/3: InfiniteTalk INT8+LoRA / FP8 T5', flush=True)
snapshot_download('MeiGen-AI/InfiniteTalk',local_dir=str(IT),
  allow_patterns=['quant_models/infinitetalk_single_int8_lora.safetensors',
   'quant_models/infinitetalk_single_int8_lora.json',
   'quant_models/t5_fp8.safetensors','quant_models/t5_map_fp8.json'],max_workers=3)
print('3/3: 音声エンコーダー', flush=True)
snapshot_download('TencentGameMate/chinese-wav2vec2-base',local_dir=str(WV),
  allow_patterns=['*.json','*.txt','*.model'], max_workers=3)
hf_hub_download('TencentGameMate/chinese-wav2vec2-base', filename='model.safetensors',
  revision='refs/pr/1',local_dir=str(WV))
print('✅ ダウンロード完了', flush=True)
'''
subprocess.run([str(VENV_PY), '-c', download_script], cwd=str(ROOT), check=True)

needed = [BASE/'config.json', BASE/'Wan2.1_VAE.pth',
 BASE/'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
 QUANT, IT/'quant_models'/'infinitetalk_single_int8_lora.json',
 IT/'quant_models'/'t5_fp8.safetensors', IT/'quant_models'/'t5_map_fp8.json',
 WV/'model.safetensors']
missing = [str(p) for p in needed if not p.is_file()]
assert not missing, '必須ファイル不足（③を再実行）:\n' + '\n'.join(missing)
for p in needed:
    print(f'✅ {p.relative_to(ROOT)} ({p.stat().st_size / 1024**2:,.1f} MiB)')


## ④ 生成設定

最初は**40秒 / 480p / 8ステップ**がおすすめ。途中確認だけなら `DURATION_SEC = 5` にしてアップロードセルを再実行すれば短いテストができます。40秒で生成するときは値を40に戻してください。手足の動きは確実ではなく、入力画像の構図やモデルの癖に依存します。

In [ ]:
# @title 生成オプション（値を書き換えられます）
DURATION_SEC = 40       # @param {type:"integer"}
SEED = 42               # @param {type:"integer"}
PROMPT = "A single anime-style young adult woman speaks naturally to the camera, accurate lip synchronization with her speech, expressive facial movements, subtle natural head movements, gentle upper-body movement and occasional small hand gestures. Keep the character identity, hair, outfit, camera angle and background consistent. No scene cuts, no extra people." # @param {type:"string"}
AUTO_DOWNLOAD = True   # @param {type:"boolean"}

assert 3 <= DURATION_SEC <= 40, 'このノートブックは3〜40秒用です。'
assert isinstance(SEED, int)
print(f'✅ {DURATION_SEC}秒 / 25fps / 480p / 8ステップ / seed={SEED}')

## ⑤ 画像と音声をアップロード（毎回実行）

PNG/JPG/WEBPから画像を1枚、WAV/MP3/M4Aなどから音声を1つ選択。**1回のダイアログで両方選択**できます。音声が40秒より長ければ先頭40秒に切り、短ければ足りない部分を無音で延長します。アップロードのたびに別の作業フォルダを使うので過去の動画は残ります。

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from pathlib import Path
import subprocess, json, datetime, uuid, shutil
from IPython.display import display, Audio

print('画像1つと音声1つを選択してください')
uploaded = files.upload()
image_ext = {'.png','.jpg','.jpeg','.webp','.bmp'}
audio_ext = {'.wav','.mp3','.m4a','.aac','.flac','.ogg','.opus','.wma','.mp4'}
imgs = [Path(x) for x in uploaded if Path(x).suffix.lower() in image_ext]
auds = [Path(x) for x in uploaded if Path(x).suffix.lower() in audio_ext]
assert len(imgs) == 1 and len(auds) == 1, f'画像1個と音声1個を選んでください（画像{len(imgs)}個・音声{len(auds)}個）'

job_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
JOB = Path('/content/infinitetalk_outputs') / job_id
JOB.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = JOB / 'reference.png'
AUDIO_PATH = JOB / 'speech_16k_mono.wav'
with Image.open(imgs[0]) as im:
    im = ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    im.save(IMAGE_PATH, optimize=True)

# 音声を正確に生成秒数へ正規化: WAV / 16kHz / mono, 無音padding
ff = ['ffmpeg','-y','-hide_banner','-loglevel','error', '-i', str(auds[0]),
      '-vn','-af', f'apad=whole_dur={DURATION_SEC}', '-t', str(DURATION_SEC),
      '-ac','1','-ar','16000','-c:a','pcm_s16le', str(AUDIO_PATH)]
subprocess.run(ff, check=True)
info = json.loads(subprocess.check_output(
    ['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(AUDIO_PATH)], text=True))
print('✅ 新しいジョブ:', JOB.name)
print('画像:', Image.open(IMAGE_PATH).size, '音声:', round(float(info['format']['duration']),2), '秒')
display(Image.open(IMAGE_PATH).resize((min(340,Image.open(IMAGE_PATH).width),
              max(1,int(Image.open(IMAGE_PATH).height * min(340,Image.open(IMAGE_PATH).width) / Image.open(IMAGE_PATH).width)))))
display(Audio(filename=str(AUDIO_PATH)))
input_json = {'prompt': PROMPT, 'cond_video': str(IMAGE_PATH),
              'cond_audio': {'person1': str(AUDIO_PATH)}}
JSON_PATH = JOB / 'input.json'
JSON_PATH.write_text(json.dumps(input_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ 準備完了。次の⑥を実行します。')

### ⑤-補足　生成前のセルフチェック（必要なときに実行）

生成前にGPUメモリ・ディスク・必要ファイルの有無を確認します。失敗原因の切り分け用。パスやサイズ以外の個人データは表示しません。

In [ ]:
from pathlib import Path
import subprocess, shutil
ROOT = Path('/content/InfiniteTalk')
BASE = ROOT / 'weights' / 'Wan2.1-I2V-14B-480P'
IT = ROOT / 'weights' / 'InfiniteTalk'
WV = ROOT / 'weights' / 'chinese-wav2vec2-base'
required = {
    'Wan config': BASE / 'config.json',
    'Wan VAE': BASE / 'Wan2.1_VAE.pth',
    'Wan CLIP': BASE / 'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
    'INT8+LoRA DiT': IT / 'quant_models' / 'infinitetalk_single_int8_lora.safetensors',
    'INT8+LoRA map': IT / 'quant_models' / 'infinitetalk_single_int8_lora.json',
    'FP8 T5': IT / 'quant_models' / 't5_fp8.safetensors',
    'FP8 T5 map': IT / 'quant_models' / 't5_map_fp8.json',
    'Audio encoder': WV / 'model.safetensors',
}
print('== GPU ==')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],check=False)
print('== ディスク ==')
print(f"{shutil.disk_usage('/content').free / 1024**3:.1f} GiB 空き")
print('== チェックポイント ==')
failed=[]
for key,path in required.items():
    ok = path.is_file() and path.stat().st_size > 0
    label = f'{path.stat().st_size / 1024**3:.2f} GiB' if ok else '見つからない / サイズ0'
    print(('✅' if ok else '❌'),key,':',label)
    if not ok: failed.append(str(path))
if failed:
    print('\nモデルファイルが不足しています。③のセルを再実行してください。')
else:
    print('\n✅ 基本ファイルは揃っています。')

## ⑥ InfiniteTalkで生成（エラー詳細を自動表示）

GPUに合わせて量子化モデルをストリーミング生成します。失敗時は、どのライブラリやモデルで落ちたかを調べるために **Tracebackや重要なエラー行を自動抽出** します。出力ログは消去されず保存されます。

**初回は④の `DURATION_SEC=5` で小さく動作確認するのを推奨します。** 失敗原因の判明後、40秒へ戻してください。

In [ ]:
from pathlib import Path
import subprocess, os, json, shutil
assert Path(JSON_PATH).exists(), '先に⑤画像と音声のアップロードを実行してください。'
assert QUANT.exists(), '先に③モデル取得を実行してください。'
RAW_PREFIX = JOB / 'infinitetalk_raw'
LOG_FILE = JOB / 'generate.log'

VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '②環境構築を実行してください。'
args = [str(VENV_PY), 'generate_infinitetalk.py',
        '--ckpt_dir', str(BASE),
        '--wav2vec_dir', str(WV),
        '--quant_dir', str(QUANT),
        '--quant', 'fp8',              # T5はfp8。INT8 DiTの復元形式は同梱quant mapから判定
        '--input_json', str(JSON_PATH),
        '--size', 'infinitetalk-480',
        '--sample_steps', '8',
        '--sample_text_guide_scale', '1.0',
        '--sample_audio_guide_scale', '2.0',
        '--sample_shift', '2',
        '--mode', 'streaming',
        '--frame_num', '81',
        '--max_frame_num', str(int(DURATION_SEC*25)),
        '--motion_frame', '9',
        '--base_seed', str(SEED),
        '--save_file', str(RAW_PREFIX),
        '--audio_save_dir', str(JOB/'audio_embeddings')]
if mem_gib < 55:
    args += ['--num_persistent_param_in_dit','0', '--t5_cpu']

# v3環境を同じColabセッションで使い回しても修復できるようにする。
# torch の拡張ロードでは Ninja が必要。uv venvにはpip自体がない場合もある。
import sys
VENV_BIN = VENV_PY.parent
if not (VENV_BIN / 'ninja').exists():
    uv_exe = shutil.which('uv')
    assert uv_exe, 'uvが見つかりません。先に②の環境構築を実行してください。'
    print('Ninjaがないため、専用環境に自動インストールします…', flush=True)
    subprocess.run([uv_exe, 'pip', 'install', '--python', str(VENV_PY), 'ninja'], check=True)

runtime_env = os.environ.copy()
runtime_env['PATH'] = str(VENV_BIN) + os.pathsep + runtime_env.get('PATH', '')
# ColabでGPU拡張の同時ビルド数が増えすぎないようにする。
runtime_env['MAX_JOBS'] = '2'
check_ninja = subprocess.run([str(VENV_BIN / 'ninja'), '--version'],
                             capture_output=True, text=True, env=runtime_env)
assert check_ninja.returncode == 0, f'Ninja が実行できません: {check_ninja.stderr}'
print('✅ Ninja:', check_ninja.stdout.strip(), flush=True)

runtime_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
runtime_env['TOKENIZERS_PARALLELISM'] = 'false'
print('生成開始（モデル読み込みと推論に時間がかかります）。出力:', RAW_PREFIX, flush=True)
with open(LOG_FILE, 'w', encoding='utf-8') as log:
    proc = subprocess.Popen(args, cwd=str(ROOT), env=runtime_env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try: proc.wait(timeout=15)
        except subprocess.TimeoutExpired: proc.kill()
        raise
if status != 0:
    # 終了コードだけでは何が失敗したか分からない。常に元のエラーを表示する。
    import re
    lines = LOG_FILE.read_text(encoding='utf-8', errors='replace').splitlines()
    # ANSIカラーコードなどを除去し、エラー解析を読みやすくする。
    clean = [re.sub(r'\x1b\[[0-9;]*m', '', line) for line in lines]
    patterns = [r'Traceback', r'Error:', r'Exception:', r'CUDA out of memory',
                r'No such file', r'not found', r'unsupported', r'AssertionError',
                r'safetensors', r'Killed', r'undefined symbol', r'cannot import',
                r'Failed', r'RuntimeError', r'KeyError']
    interesting = [i for i, line in enumerate(clean)
                   if any(re.search(pat, line, re.I) for pat in patterns)]
    print('\n'+'='*24+' InfiniteTalk 失敗診断 '+'='*24)
    print('終了コード:', status)
    print('ログ:', LOG_FILE)
    if interesting:
        print('\n▼ 検出したエラー箇所（前後の文脈を含む）')
        a, b = max(0,interesting[-1]-24), min(len(clean),interesting[-1]+24)
        print('\n'.join(f'{i+1}: {clean[i]}' for i in range(a,b)))
    print('\n▼ ログ末尾（最大100行）')
    print('\n'.join(clean[-100:]))
    try:
        gpu_report = subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.free,memory.total',
                                     '--format=csv,noheader'], capture_output=True,text=True)
        print('\nGPU 状態:',gpu_report.stdout.strip() or gpu_report.stderr.strip())
    except Exception as ex:
        print('GPU診断不可:',ex)
    text_lower = '\n'.join(clean[-200:]).lower()
    if any(x in text_lower for x in ('out of memory','cuda error: out of memory', 'cuda oom')):
        print('\n推奨：GPUメモリ不足の可能性が高いです。5秒テスト、省VRAM、G4/A100 80GBを検討。')
    elif 'ninja is required' in text_lower:
        print('\n推奨：Ninjaが見つかりません。②を再実行するかv4ノートブックで再実行してください。')
    elif any(x in text_lower for x in ('nvcc', 'cuda_home', 'cuda toolkit')):
        print('\n推奨：CUDA拡張のビルドでエラー。ログ内のCUDA compiler / nvcc の記述を確認してください。')
    elif any(x in text_lower for x in ('no such file', 'filenotfounderror','entry not found')):
        print('\n推奨：モデルやファイルの欠損が考えられます。③のダウンロードを再実行してください。')
    elif any(x in text_lower for x in ('modulenotfounderror','importerror','undefined symbol')):
        print('\n推奨：ライブラリの互換性問題が考えられます。ログのエラー行を共有してください。')
    elif any(x in text_lower for x in ('no space left on device', 'enospc')):
        print('\n推奨：Colabのディスク空き容量を確認してください。')
    else:
        print('\n推奨：根本原因は終了コードだけでは不明です。上のログを共有してください。')
    raise RuntimeError(f'InfiniteTalk生成失敗（終了コード {status}）。上に実際のエラーとログ末尾を表示しました。')
RAW_VIDEO = Path(str(RAW_PREFIX) + '.mp4')
assert RAW_VIDEO.exists() and RAW_VIDEO.stat().st_size > 10_000, f'出力動画がありません: {RAW_VIDEO}'
print('✅ 推論完了:', RAW_VIDEO)

## ⑦ 動画の確認・MP4ダウンロード

生成映像の音声を正規化済み音声に合わせ、フレーム末尾が短い場合だけ最終フレームを延長して指定秒数に揃えます。**生成された実際の動きが40秒分あるとは限らない**ため、最終結果を必ず確認してください。

In [ ]:
from IPython.display import Video, display, FileLink
from google.colab import files
import subprocess, json
assert RAW_VIDEO.exists(), '先に⑥を実行してください。'
FINAL_VIDEO = JOB / f'InfiniteTalk_{DURATION_SEC}s_{job_id}.mp4'
# 音声はアップロードされたものを再使用。映像の末尾が短いときだけホールド。
ffcmd = ['ffmpeg','-y','-hide_banner','-loglevel','error',
         '-i',str(RAW_VIDEO), '-i',str(AUDIO_PATH),
         '-filter_complex', f'[0:v]fps=25,tpad=stop_mode=clone:stop_duration=3,trim=duration={DURATION_SEC},setpts=PTS-STARTPTS[v]',
         '-map','[v]','-map','1:a:0',
         '-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p',
         '-c:a','aac','-b:a','192k','-t',str(DURATION_SEC), '-movflags','+faststart', str(FINAL_VIDEO)]
subprocess.run(ffcmd, check=True)
details = json.loads(subprocess.check_output(
    ['ffprobe','-v','error','-show_entries','format=duration,size','-of','json',str(FINAL_VIDEO)], text=True))['format']
print('✅ 完成:', FINAL_VIDEO)
print('長さ:',round(float(details['duration']),2),'秒 /', round(int(details['size'])/1024**2, 1), 'MiB')
display(Video(filename=str(FINAL_VIDEO), embed=True, width=480))
display(FileLink(str(FINAL_VIDEO)))
if AUTO_DOWNLOAD:
    files.download(str(FINAL_VIDEO))
print('再生成: 画像/音声を替えるなら⑤→⑥→⑦を再実行。設定だけ変更する場合は④→⑤→⑥→⑦。')

---
### エラーと対応

- **pipのCalledProcessError / Python 3.13**：前版の環境②は使わず、このv4の②を実行。専用Python 3.10で構築します。エラー時はセル末尾にuvが実際のパッケージ原因を出力します。
- **`Ninja is required to load C++ extensions`**：v4ではNinjaを自動導入し、生成プロセスのPATHも修正します。旧v3なら専用環境へ`uv pip install --python /content/infinitetalk_py310/bin/python ninja`で導入できます。
- **`nvcc` / CUDA拡張のビルド失敗**：Ninja導入後にCUDA Toolkitの有無やCUDAバージョン不一致が顕在化する場合があります。生成ログを確認してください。
- **`CUDA out of memory`**：G4またはA100 80GBで試す。A100 40GBではCPU退避しますが速度と成功は保証されません。まず5秒に短縮してください。
- **`No space left on device`**：①で空きディスクを確認し、不要モデルや古い動画を消してください。
- **`import ...` のエラー**：②を再実行し、依存関係の具体的なログを確認してください。③より先に②を実行してください。
- **`AssertionError: FLASH_ATTN_2_AVAILABLE`**：②-BでSDPA代替パッチを適用し、GPU動作テストが成功したら⑥を実行。FlashAttention 2のビルドは不要です。
- **モデルダウンロードが途中で切れた**：③を再実行してください。
- **音声とのズレ/手が不自然**：元画像・音声・seedを変えて短時間試験し、改善後40秒で生成してください。

### 注意

本修正版は静的チェック済みですが、実GPUでの40秒推論成功を保証するものではありません。生成モデルや依存ライブラリの公開側が変更されると動作しなくなることがあります。ログの最後の30行とGPU名を見せてください。

**参考**：[InfiniteTalk公式](https://github.com/MeiGen-AI/InfiniteTalk)、[量子化重み](https://huggingface.co/MeiGen-AI/InfiniteTalk/tree/main/quant_models)


### ログが必要な場合

Colabの新規コードセルで `!tail -n 100 /content/infinitetalk_outputs/*/generate.log` を実行してください。複数ログがあるときは最新のジョブを選んでください。